# QGFD Paper Experiments — single-GPU driver

Runs every track of the QGFD introductory paper end to end and emits
`paper/REPORT.md` with populated tables. Sized for a **free T4 / P100 (~16 GB)**;
runs unchanged on Kaggle (Internet ON) or Colab.

| Track | Script | Cost per model |
| --- | --- | --- |
| 1. Zero-shot: perplexity, noise robustness, attention stats, latency | `scripts/review_experiments.py` | ~5–15 min for 3 seeds |
| 2. LoRA fine-tuning A/B (LoRA-only vs LoRA+QGFD) | `scripts/finetune_qgfd.py` | ~20–60 min for 3 seeds |
| 3. Synthetic multi-hop: induction + passkey | `scripts/eval_synthetic.py` | ~3–10 min for 3 seeds |
| 4. Ablation + the α=0 equivalence check (E1) | inline below | ~10 min |
| 5. Mechanism falsifiers E2–E8 | `scripts/mechanism_experiments.py` | ~10–25 min, +20 min on the smallest |
| 6. E9 denominator control | derived in `scripts/build_report.py` | free — re-reads Track 1 |

**Total: a few GPU-hours.** Each track writes its own aggregate JSON, and the
report builder ingests whatever exists — so you can stop after any track and
still get a coherent (partial) report. Every track is behind a `RUN_*` toggle;
if a Kaggle session times out, switch off what already finished and resume.

**What the headline claim is.** Zero-shot QGFD is expected to be
perplexity-neutral-to-slightly-worse on clean text. The claim is the *robustness
gap*: QGFD's perplexity should degrade **less** under input noise. Every number
carries a t-based 95% CI over seeds, and the paired (within-seed) statistic is the
one to trust.

**What could kill it.** Track 5 exists to give that question a real answer rather
than a hopeful one. Its E3 arm matches attention entropy with a plain temperature
rescale — free at inference, and foldable into `W_Q` — and if that reproduces the
robustness gap, there is no mechanism contribution left to report. Run Track 5
before writing any prose about *why* QGFD works.

**The cheapest falsifier is E9, and it needs no GPU at all.** The headline
statistic is a difference of *relative* degradations, `100·(noisy−clean)/clean`
per arm. QGFD's clean perplexity is higher, so its denominator is larger — and a
larger denominator shrinks its Δ% *at identical accuracy under noise*. The report
splits the gap into that arithmetic and the residual (Table 1b); the residual is
positive exactly when QGFD's **absolute** perplexity under noise is lower. If the
residual does not clear zero, there is no robustness result to report, and the
abstract retracts contribution (2) automatically.

In [ ]:
# --- Environment -------------------------------------------------------------
# Kaggle: turn Internet ON in the notebook settings panel before running this
# (Settings -> Internet). Without it the clone, the pip installs and the
# WikiText-2 download all fail. Accelerator: any single T4 or P100 is enough;
# a T4 x2 instance works too but only cuda:0 is used.
import os, subprocess, sys

REPO_URL, REPO_DIR = "https://github.com/rajboopathiking/TorchDire.git", "TorchDire"
if not os.path.isdir("torchdire"):                 # not already inside the repo
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "peft>=0.10", "datasets>=2.18",
                "accelerate", "matplotlib"], check=False)

In [ ]:
# --- GPU and dtype -----------------------------------------------------------
# bf16 *tensor cores* need compute capability >= 8.0, and a T4 is sm_75 / a P100 is
# sm_60 — yet torch.cuda.is_bf16_supported() returns True on a T4 anyway, because
# torch emulates bf16 there. It runs, and it is numerically SAFER than fp16 (same
# exponent range as fp32, so no overflow in the perplexity accumulations); it is
# just slower. Both arms use the same dtype either way, so this is never a
# confound — but the paper must report which one ran, so print both signals.
import torch

BF16_OVERRIDE = None      # None = auto; set True/False to pin the dtype explicitly

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    has_tensor_cores = p.major >= 8
    torch_says = torch.cuda.is_bf16_supported()
    use_bf16 = torch_says if BF16_OVERRIDE is None else BF16_OVERRIDE
    DTYPE = "bfloat16" if use_bf16 else "float16"
    print(f"{p.name} | sm_{p.major}{p.minor} | {p.total_memory / 2**30:.1f} GiB "
          f"| {torch.cuda.device_count()} device(s)")
    print(f"bf16 tensor cores (sm>=80): {has_tensor_cores} | "
          f"torch.cuda.is_bf16_supported(): {torch_says}")
    if use_bf16 and not has_tensor_cores:
        print("  -> EMULATED bf16 on pre-Ampere hardware: correct but slower than "
              "fp16. Report the dtype as bfloat16.")
    if torch.cuda.device_count() > 1:
        print("  -> multi-GPU (e.g. Kaggle T4 x2). Everything here resolves to cuda:0 "
              "only — no sharding, so the latency numbers stay single-device.")
else:
    DTYPE = "float32"
    print("No CUDA device — falling back to CPU/float32. Set QUICK = True below.")
print("torch", torch.__version__, "| dtype for this run:", DTYPE)

## Configuration

`MODELS` are the three ungated SLMs a 16 GB card can hold. `TinyLlama-1.1B` has
real GQA (32 heads / 4 KV) and is the headline model; `Qwen2.5-0.5B` is the
cross-family check (qwen2 adapter rather than llama).

Set `QUICK = True` for a ~5-minute end-to-end rehearsal on the smallest model
before committing to the full run.

In [ ]:
# --- Configuration -----------------------------------------------------------
QUICK = False          # True => tiny rehearsal on one model, minutes not hours

MODELS = [
    "HuggingFaceTB/SmolLM2-135M",              # llama, 135M
    "Qwen/Qwen2.5-0.5B",                       # qwen2, 0.5B — cross-family
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",      # llama, 1.1B, real GQA — headline
]
SEEDS = (0, 1, 2)
RESULTS = "qgfd_paper_results"

RUN_ZEROSHOT, RUN_FINETUNE, RUN_SYNTHETIC, RUN_ABLATION = True, True, True, True
RUN_MECHANISM = True   # Track 5, the E1–E8 falsifiers — see that section

# Track 5 routing. E2/E3/E4/E6 are perplexity-only and cheap enough for every
# model. E7 re-runs both retrieval probes at five depths and E8 loads a third
# checkpoint, so both are restricted to the smallest model.
MECH_ALL_MODELS = ("E2", "E3", "E4", "E6")
MECH_SMALLEST_ONLY = ("E7", "E8")
MECH_LARGE_MODEL = "HuggingFaceTB/SmolLM2-360M"   # E8's "next size up"

if QUICK:
    MODELS, SEEDS = ["JackFram/llama-160m"], (0, 1)
    RESULTS = "qgfd_quick_results"
    MECH_LARGE_MODEL = "HuggingFaceTB/SmolLM2-135M"

os.makedirs(RESULTS, exist_ok=True)
print(f"{len(MODELS)} model(s) x {len(SEEDS)} seed(s) -> {RESULTS}/")
for m in MODELS:
    print("  ", m)
print("tracks:", ", ".join(n for n, on in
      [("zeroshot", RUN_ZEROSHOT), ("finetune", RUN_FINETUNE),
       ("synthetic", RUN_SYNTHETIC), ("ablation", RUN_ABLATION),
       ("mechanism", RUN_MECHANISM)] if on) or "none")

## Track 1 — Zero-shot: perplexity, noise robustness, attention, latency

Both arms run through the *same* adapter; the only difference is whether `p` comes
from `SoftmaxOperator` or `QGFDOperator`. `verify_patch()` raises if the patch
silently no-ops, so a green run means the operator was genuinely reached.

The baseline is **eager materialised softmax**, not SDPA/FlashAttention — QGFD
needs the explicit probability matrix. Read every latency figure that way.

In [4]:
# --- Track 1 -----------------------------------------------------------------
import traceback
from dataclasses import replace

from scripts.review_experiments import ExperimentConfig, run_all_seeds

zeroshot = {}
if RUN_ZEROSHOT:
    for mid in MODELS:
        cfg = ExperimentConfig(
            model_id=mid, dtype=DTYPE, device="auto",
            diffusion_steps=1, target_alpha=0.05,
            out_dir=f"{RESULTS}/zeroshot/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = replace(cfg, ppl_num_texts=8, robustness_num_texts=6,
                          attn_num_texts=2, ppl_max_length=128, ppl_stride=128,
                          text_pool_size=64, latency_seq_len=128, latency_iters=3,
                          latency_warmup=1, gen_max_new_tokens=8)
        try:
            zeroshot[mid] = run_all_seeds(cfg, seeds=SEEDS)
        except Exception:
            traceback.print_exc()
            print(f"!! zero-shot FAILED for {mid} — continuing with the other models")
print("zero-shot done:", list(zeroshot))


##########################################################################
# SEED 0  (1/3)
##########################################################################
Loading WikiText-2 (pool 1200, subsampling 200 @ text_sample_seed=0) ...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

  [qgfd] patch verified: 22 x LlamaAttentionAdapter, operator invoked 22x
  [qgfd] perplexity ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (37043 > 2048). Running this sequence through the model will result in indexing errors


  [qgfd] robustness sweep ...
  [qgfd] attention stats ...
  [qgfd] latency ...
  [qgfd] generation ...


[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Saved raw results -> qgfd_paper_results/zeroshot/TinyLlama-1.1B-Chat-v1.0/seed2/results.json

QGFD REVIEW — SUMMARY  (softmax baseline vs QGFD)
Metric                                  softmax         qgfd
--------------------------------------------------------------
Clean perplexity                         13.639       13.701
Attn entropy (nats)                       1.379        1.422
Sink mass @ pos0                         0.6314       0.6299
Prefill latency (ms)                     525.19       677.93
Tokens / s                                974.9        755.2
--------------------------------------------------------------
QGFD compute overhead: 1.291x baseline prefill

Robustness (perplexity vs noise rate; Δ% vs clean):
    rate   softmax_ppl    sm_Δ%    qgfd_ppl    qg_Δ%
    0.00        13.619      0.0      13.694      0.0
    0.05        53.780    294.9      54.176    295.6
    0.10       112.644    727.1     113.436    728.4
    0.15       180.794   1227.5     181.984   1228.

## Track 2 — LoRA fine-tuning A/B

Equal budget, identical adapters on `q/k/v/o`, identical seed / data / LR /
schedule / step count. The only difference is the probability operator, so the
comparison isolates QGFD rather than adapter capacity.

Two guards worth knowing about, because both used to fail silently:

* `verify_lora_live()` runs a probe forward+backward and **refuses to report a
  result** unless a `lora_B` actually receives gradient. The adapter aliases
  `q/k/v/o` onto itself, and `nn.Module.named_modules()` de-duplicates shared
  submodules — so PEFT used to inject LoRA into a module `forward()` never called.
* α warmup is driven by a `TrainerCallback`, and `report_alpha()` checks that
  `step_count` actually advanced past `warmup_steps`. Mutating step state inside
  `forward()` diverges on gradient-checkpoint recompute.

If you are tight on VRAM, lower `batch_size` and raise `grad_accum` — the product
is what matters.

In [ ]:
# --- torchao: only needed to make `Trainer` importable on some images ---------
# transformers' Trainer imports torchao when the installed version advertises
# quantization support. Colab/Kaggle images ship a torchao that predates the
# local torch build, and the import raises. Upgrading fixes the import; the
# "Skipping import of cpp extensions due to incompatible torch version" warning
# that follows is expected and harmless — nothing here uses torchao's kernels.
# Skipped entirely if Trainer already imports, so a clean image is left alone.
import subprocess, sys

try:
    from transformers import Trainer  # noqa: F401
    print("transformers.Trainer imports cleanly — no torchao change needed.")
except Exception as exc:                                   # noqa: BLE001
    print(f"Trainer import failed ({type(exc).__name__}) — upgrading torchao ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchao==0.16.0"],
                   check=False)
    print("Upgraded. RESTART THE KERNEL and re-run from the top if the next cell "
          "still cannot import Trainer.")

In [ ]:
# --- Track 2 -----------------------------------------------------------------
from scripts.finetune_qgfd import FinetuneConfig, apply_quick
from scripts.finetune_qgfd import run_all_seeds as ft_run_all_seeds

finetune = {}
if RUN_FINETUNE:
    for mid in MODELS:
        cfg = FinetuneConfig(
            model_id=mid, dtype=DTYPE, device="auto", backend="operator",
            diffusion_steps=1, target_alpha=0.05, warmup_steps=100,
            max_steps=300, batch_size=2, grad_accum=8, block_size=256,
            learning_rate=2e-4, gradient_checkpointing=True,
            out_dir=f"{RESULTS}/finetune/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = apply_quick(cfg)
        try:
            finetune[mid] = ft_run_all_seeds(cfg, seeds=SEEDS)
        except Exception:
            traceback.print_exc()
            print(f"!! fine-tune FAILED for {mid} — continuing with the other models")
print("fine-tune done:", list(finetune))


##########################################################################
# FINETUNE SEED 0  (1/3)
##########################################################################
Loading WikiText-2 train split (1500 paragraphs) ...
  (corpus: Salesforce/wikitext / wikitext-2-raw-v1, 15815 usable paragraphs)
Loading WikiText-2 test split (60 paragraphs) ...
  (corpus: Salesforce/wikitext / wikitext-2-raw-v1, 1889 usable paragraphs)

--- arm: softmax (backend=operator) ---


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [softmax] 30 x LlamaAttentionAdapter, 1 operator instance(s)


  LoRA live: 240 trainable tensors, 120 with non-zero grad on the probe


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (250374 > 8192). Running this sequence through the model will result in indexing errors


...

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [qgfd] 30 x LlamaAttentionAdapter, 1 operator instance(s)
  LoRA live: 240 trainable tensors, 120 with non-zero grad on the probe


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (250374 > 8192). Running this sequence through the model will result in indexing errors


  train data: 978 blocks x 256 tokens = 250368 tokens
  (TrainingArguments: dropped unsupported ['warmup_ratio'])


Step,Training Loss
10,3.307766
20,3.198897
30,3.206562
40,3.092969
50,3.079721
60,3.042284


## Track 3 — Synthetic multi-hop probes

**Induction.** A random word sequence `S` is shown twice; at each position of the
second copy the model must emit whatever followed the same word in the first copy.
That is the canonical two-hop circuit. A fraction of the second copy can be
replaced by unrelated words (`induction_noise_rates`); corrupted positions are
excluded from scoring, so the remaining ones stay well-posed but must route
through a garbled context — the same axis as the headline robustness claim.

**Passkey.** A 5-digit key buried at a controlled depth in filler text, retrieved
at the end. Decoding uses an explicit greedy loop with `use_cache=False`, so the
probe exercises exactly the operator the model was trained with.

`control_acc` scores the **first** copy, where the answer is unpredictable. It is
the chance-level floor: if induction accuracy is not far above it, the row is
uninformative no matter which arm wins.

In [6]:
# --- Track 3 -----------------------------------------------------------------
from scripts.eval_synthetic import SyntheticConfig
from scripts.eval_synthetic import apply_quick as syn_quick
from scripts.eval_synthetic import run_all_seeds as syn_run_all_seeds

synthetic = {}
if RUN_SYNTHETIC:
    for mid in MODELS:
        cfg = SyntheticConfig(
            model_id=mid, dtype=DTYPE, device="auto", backend="operator",
            diffusion_steps=1, target_alpha=0.05,
            induction_num_examples=64, induction_seq_len=48,
            induction_noise_rates=(0.0, 0.2, 0.4),
            passkey_num_examples=24, passkey_context_tokens=384,
            out_dir=f"{RESULTS}/synthetic/{mid.split('/')[-1]}",
        )
        if QUICK:
            cfg = syn_quick(cfg)
        try:
            synthetic[mid] = syn_run_all_seeds(cfg, seeds=SEEDS, post_lora=False)
        except Exception:
            traceback.print_exc()
            print(f"!! synthetic FAILED for {mid} — continuing with the other models")
print("synthetic done:", list(synthetic))


##########################################################################
# SEED 0  (1/3)
##########################################################################

--- arm: softmax (zero-shot) ---


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [softmax] 30 x LlamaAttentionAdapter, 1 operator instance(s)
  [softmax] induction ...
  [softmax] induction acc = 0.9524 (control 0.0000)
    ctx corruption 0.20: acc = 0.8093 (n=2333)
    ctx corruption 0.40: acc = 0.5405 (n=1789)
  [softmax] passkey ...
  [softmax] passkey acc  = 1.0000 @ 427 ctx tokens

--- arm: qgfd (zero-shot) ---


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [qgfd] 30 x LlamaAttentionAdapter, 1 operator instance(s)
  [qgfd] induction ...
  [qgfd] induction acc = 0.9392 (control 0.0000)
    ctx corruption 0.20: acc = 0.7861 (n=2333)
    ctx corruption 0.40: acc = 0.5109 (n=1789)
  [qgfd] passkey ...
  [qgfd] passkey acc  = 1.0000 @ 427 ctx tokens

Saved synthetic results -> qgfd_paper_results/synthetic/SmolLM2-135M/seed0/synthetic_results.json

SYNTHETIC MULTI-HOP — seed 0  [zero-shot]
metric                                       softmax              qgfd
Induction acc (clean)                         0.9524            0.9392
  (control: first copy)                       0.0000            0.0000
  induction @ ctx noise 0.20                  0.8093            0.7861
  induction @ ctx noise 0.40                  0.5405            0.5109
Passkey acc (strict)                          1.0000            1.0000
  passkey @ depth 0.10                        1.0000            1.0000
  passkey @ depth 0.50                        1.0000            1.

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  [qgfd] 24 x Qwen2AttentionAdapter, 1 operator instance(s)
  [qgfd] induction ...
  [qgfd] induction acc = 0.9976 (control 0.0000)
    ctx corruption 0.20: acc = 0.9643 (n=2327)
    ctx corruption 0.40: acc = 0.8435 (n=1776)
  [qgfd] passkey ...
  [qgfd] passkey acc  = 1.0000 @ 425 ctx tokens

Saved synthetic results -> qgfd_paper_results/synthetic/Qwen2.5-0.5B/seed2/synthetic_results.json

SYNTHETIC MULTI-HOP — seed 2  [zero-shot]
metric                                       softmax              qgfd
Induction acc (clean)                         0.9986            0.9976
  (control: first copy)                       0.0000            0.0000
  induction @ ctx noise 0.20                  0.9665            0.9643
  induction @ ctx noise 0.40                  0.8497            0.8435
Passkey acc (strict)                          1.0000            1.0000
  passkey @ depth 0.10                        1.0000            1.0000
  passkey @ depth 0.50                        1.0000            1.

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [softmax] 22 x LlamaAttentionAdapter, 1 operator instance(s)
  [softmax] induction ...
!! synthetic FAILED for TinyLlama/TinyLlama-1.1B-Chat-v1.0 — continuing with the other models
synthetic done: ['HuggingFaceTB/SmolLM2-135M', 'Qwen/Qwen2.5-0.5B']


Traceback (most recent call last):
  File "/tmp/ipykernel_2093/832202123.py", line 20, in <cell line: 0>
    synthetic[mid] = syn_run_all_seeds(cfg, seeds=SEEDS, post_lora=False)
                     ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/TorchDire/scripts/eval_synthetic.py", line 618, in run_all_seeds
    runs.append(run_seed(
                ~~~~~~~~^
        replace(cfg, seed=s, out_dir=os.path.join(cfg.out_dir, f"seed{s}")),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        arms=arms, post_lora=post_lora))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/TorchDire/scripts/eval_synthetic.py", line 449, in run_seed
    "arms": {a: eval_arm(a, cfg, post_lora, train_texts, eval_texts)
                ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/TorchDire/scripts/eval_synthetic.py", line 377, in eval_arm
    induction = eval_induction(model, tok, device, cfg)
  File "/content/TorchDire/scr

## Track 4 — Ablation and the α=0 equivalence check

Smallest model only, to save compute. Two separate things happen here.

**The equivalence check is the important one.** Contribution (1) of the paper is
that QGFD at α=0 is *exactly* softmax. That is a falsifiable claim about the
implementation, and it costs one forward pass to test: if `QGFDOperator(α=0)` and
`SoftmaxOperator` do not give bit-identical logits, the drop-in claim is false and
nothing else in the paper should be believed.

**Then the grid:** `T ∈ {1,2}` × `α ∈ {0.02, 0.05}` × `detach_P ∈ {True, False}`,
one seed each, reporting clean perplexity and degradation at the highest noise
rate. This is for *direction* only — one seed cannot separate settings that differ
by ~1%. (`experiments/ablation.py`'s `QGFDAblator` reports ROUGE/BLEU/BERTScore
from hard-coded heuristics and `QGFDProfiler`'s FLOPs/VRAM are analytic estimates;
neither is used here or anywhere in the report.)

In [ ]:
# --- E1: α=0 must be exactly softmax -----------------------------------------
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from torchdire import QGFDOperator, SoftmaxOperator, wrap_model_with_qgfd_operator

_MID = MODELS[0]
_tok = AutoTokenizer.from_pretrained(_MID)
_ids = _tok("Diffusion over the key graph should vanish when alpha is zero.",
            return_tensors="pt")["input_ids"]

_logits = {}
for _name, _op in (("softmax", SoftmaxOperator()),
                   ("qgfd_alpha0", QGFDOperator(diffusion_steps=1, target_alpha=0.0,
                                                warmup_steps=0, detach_P=True,
                                                mode="full", is_causal=True))):
    _m = AutoModelForCausalLM.from_pretrained(_MID, torch_dtype=torch.float32)
    torch.manual_seed(0)
    _m = wrap_model_with_qgfd_operator(_m, _op, verbose=False).eval()
    with torch.no_grad():
        _logits[_name] = _m(input_ids=_ids).logits.clone()
    del _m

_delta = (_logits["softmax"] - _logits["qgfd_alpha0"]).abs().max().item()
print(f"max |logit difference| at alpha=0: {_delta:.3e}")

# Written to disk so the E1–E8 coverage preflight below can see that E1 ran, and
# so the number is recoverable after a kernel restart rather than only in a cell
# output that a re-run would overwrite.
os.makedirs(RESULTS, exist_ok=True)
with open(f"{RESULTS}/equivalence.json", "w") as fh:
    json.dump({"experiment": "E1", "model_id": _MID, "dtype": "float32",
               "max_abs_logit_delta": _delta, "tolerance": 1e-5,
               "passed": _delta < 1e-5}, fh, indent=2)

assert _delta < 1e-5, ("alpha=0 is NOT equivalent to softmax — the drop-in claim "
                       "is false, fix the operator before reporting anything else")
print("PASS: QGFD at alpha=0 reproduces softmax.")

In [8]:
# --- Ablation grid (one seed, direction only) --------------------------------
import itertools, json

from scripts.review_experiments import run_all

ablation = []
if RUN_ABLATION:
    _base = ExperimentConfig(
        model_id=MODELS[0], dtype=DTYPE, device="auto",
        ppl_num_texts=60, robustness_num_texts=40, attn_num_texts=8,
        latency_seq_len=256, latency_iters=5, gen_max_new_tokens=8,
        seed=0, robustness_seed=0, text_sample_seed=0,
    )
    if QUICK:
        _base = replace(_base, ppl_num_texts=6, robustness_num_texts=4,
                        attn_num_texts=2, ppl_max_length=128, ppl_stride=128,
                        text_pool_size=64, latency_seq_len=128, latency_iters=2)

    for T, alpha, det in itertools.product((1, 2), (0.02, 0.05), (True, False)):
        tag = f"T{T}_a{alpha}_detach{det}"
        cfg = replace(_base, diffusion_steps=T, target_alpha=alpha, detach_P=det,
                      out_dir=f"{RESULTS}/ablation/{tag}")
        try:
            r = run_all(cfg)
            qg = r["arms"]["qgfd"]
            rate = max(float(k) for k in qg["robustness"])
            base = qg["robustness"][min(qg["robustness"], key=float)]
            worst = qg["robustness"][max(qg["robustness"], key=float)]
            ablation.append({"T": T, "alpha": alpha, "detach_P": det,
                             "clean_ppl": qg["clean_ppl"],
                             "noise_rate": rate,
                             "degradation_pct": 100.0 * (worst - base) / base})
        except Exception:
            traceback.print_exc()
            print(f"!! ablation FAILED for {tag}")

    with open(f"{RESULTS}/ablation/ablation.json", "w") as fh:
        json.dump(ablation, fh, indent=2)
    print(f"\n{'T':>2} {'alpha':>6} {'detach_P':>9} {'clean PPL':>11} {'degr %':>9}")
    for row in ablation:
        print(f"{row['T']:>2} {row['alpha']:>6} {str(row['detach_P']):>9} "
              f"{row['clean_ppl']:>11.4f} {row['degradation_pct']:>9.1f}")
    print("\nOne seed per row — read direction, not differences.")

Loading WikiText-2 (pool 1200, subsampling 60 @ text_sample_seed=0) ...
  (corpus: Salesforce/wikitext / wikitext-2-raw-v1, 1889 usable paragraphs)

=== Arm: softmax ===


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [softmax] patch verified: 30 x LlamaAttentionAdapter, operator invoked 30x
  [softmax] perplexity ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9940 > 8192). Running this sequence through the model will result in indexing errors


  [softmax] robustness sweep ...
  [softmax] attention stats ...
  [softmax] latency ...
  [softmax] generation ...

=== Arm: qgfd ===


...

  [softmax] robustness sweep ...
  [softmax] attention stats ...
  [softmax] latency ...
  [softmax] generation ...

=== Arm: qgfd ===


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  [qgfd] patch verified: 30 x LlamaAttentionAdapter, operator invoked 30x
  [qgfd] perplexity ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9940 > 8192). Running this sequence through the model will result in indexing errors


  [qgfd] robustness sweep ...
  [qgfd] attention stats ...
  [qgfd] latency ...
  [qgfd] generation ...

Saved raw results -> qgfd_paper_results/ablation/T2_a0.05_detachFalse/results.json

QGFD REVIEW — SUMMARY  (softmax baseline vs QGFD)
Metric                                  softmax         qgfd
--------------------------------------------------------------
Clean perplexity                         29.850       30.510
Attn entropy (nats)                       1.587        1.677
Sink mass @ pos0                         0.4295       0.4298
Prefill latency (ms)                      55.38        75.81
Tokens / s                               4622.5       3377.0
--------------------------------------------------------------
QGFD compute overhead: 1.369x baseline prefill

Robustness (perplexity vs noise rate; Δ% vs clean):
    rate   softmax_ppl    sm_Δ%    qgfd_ppl    qg_Δ%
    0.00        29.555      0.0      30.268      0.0
    0.05       151.421    412.3     153.597    407.5
    0.10  

## Compute overhead vs sequence length

Prefill latency and peak VRAM at L ∈ {128, 256, 512}, per arm. QGFD adds a `K·Kᵀ`
GEMM plus one `p·P` product per diffusion step, so overhead should grow with L —
but the dominant cost is structural, not arithmetic: materialising `p` forecloses
fused attention kernels entirely. Against a FlashAttention baseline the gap would
be larger than anything measured here.

In [9]:
# --- Latency / VRAM vs sequence length ---------------------------------------
from scripts.review_experiments import benchmark_latency, make_model

overhead = []
_mid = MODELS[-1] if not QUICK else MODELS[0]
for _arm in ("softmax", "qgfd"):
    _cfg = ExperimentConfig(model_id=_mid, dtype=DTYPE, device="auto",
                            latency_iters=10 if not QUICK else 2,
                            latency_warmup=3 if not QUICK else 1)
    _tokL, _mL, _dev = make_model(_arm, _cfg)
    for L in ((128, 256, 512) if not QUICK else (128,)):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        r = benchmark_latency(_mL, _tokL, _dev, replace(_cfg, latency_seq_len=L))
        overhead.append({"arm": _arm, "seq_len": L, "prefill_ms": r["prefill_ms"],
                         "peak_vram_mb": (torch.cuda.max_memory_allocated() / 2**20
                                          if torch.cuda.is_available() else None)})
    del _mL
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'arm':>8} {'L':>5} {'prefill ms':>11} {'peak VRAM MB':>13}")
for r in overhead:
    v = f"{r['peak_vram_mb']:.0f}" if r["peak_vram_mb"] else "n/a"
    print(f"{r['arm']:>8} {r['seq_len']:>5} {r['prefill_ms']:>11.2f} {v:>13}")
for L in sorted({r["seq_len"] for r in overhead}):
    s = next(r for r in overhead if r["arm"] == "softmax" and r["seq_len"] == L)
    q = next(r for r in overhead if r["arm"] == "qgfd" and r["seq_len"] == L)
    print(f"L={L}: QGFD is {q['prefill_ms'] / s['prefill_ms']:.2f}x eager softmax")

with open(f"{RESULTS}/overhead.json", "w") as fh:
    json.dump({"model_id": _mid, "dtype": DTYPE, "rows": overhead}, fh, indent=2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [softmax] patch verified: 22 x LlamaAttentionAdapter, operator invoked 22x


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [qgfd] patch verified: 22 x LlamaAttentionAdapter, operator invoked 22x
     arm     L  prefill ms  peak VRAM MB
 softmax   128      140.04          2121
 softmax   256      259.26          2141
 softmax   512      528.74          2213
    qgfd   128      152.00          2131
    qgfd   256      296.41          2179
    qgfd   512      682.73          2357
L=128: QGFD is 1.09x eager softmax
L=256: QGFD is 1.14x eager softmax
L=512: QGFD is 1.29x eager softmax


## Track 5 — Mechanism falsifiers (E1–E9)

Tracks 1–4 measure *whether* QGFD changes anything. Track 5 asks *why*, and every
experiment here is built so that a specific answer would end a specific claim. Run
it and the whole picture is on one screen.

| # | Experiment | Where it runs | A negative result kills |
| --- | --- | --- | --- |
| **E1** | α=0 is bit-exact softmax | Track 4 cell above | contribution (1), the drop-in claim |
| **E2** | α × noise sweep + curvature exponent `k` | here, all models | the theory's account of the clean-PPL cost (predicts `k≈2`) |
| **E3** | entropy-matched **temperature** control | here, all models | **the entire robustness contribution** |
| **E4** | P-structure ablation: `real` / `uniform` / `shuffled` | here, all models | the word "graph" in the method name |
| **E5** | multi-seed paired robustness gap | Track 1 | the headline itself |
| **E6** | corruption types: `char` / `word_drop` / `word_swap` | here, all models | the breadth of "input noise" |
| **E7** | diffusion depth `T ∈ {1,2,3,4}` on retrieval probes | here, smallest only | the useful-reach story for `T > 1` |
| **E8** | iso-compute: small+QGFD vs the next size up | here, smallest only | any efficiency framing |
| **E9** | denominator control on the headline statistic | report builder, **free** | the headline itself, more directly than E5 |

**E9 is free, so read it first.** It re-reads the per-seed `values` already in
`results_aggregated.json` — no model, no GPU, no new run. Because
`robustness_gap_pct` divides by each arm's *own* clean perplexity, QGFD's
clean-PPL cost inflates the gap by `100·noisy_sm·(1/clean_sm − 1/clean_qgfd)`
before any robustness enters. Subtract that and what is left is
`100·(noisy_sm − noisy_qgfd)/clean_qgfd`, which is positive iff QGFD is
genuinely more accurate on corrupted text. Note the direction of the confound:
the measured clean costs are *monotone* with the measured gaps, which is exactly
what a denominator artefact looks like.

**E3 is the one to read next.** `softmax(scores/τ)` changes attention entropy at
zero compute and zero memory, and it **folds into `W_Q`** — so the pretrained model
already had both the capacity and the gradient signal to pick any τ it wanted. This
cell bisects τ until its mean attention entropy matches QGFD's, then runs the same
corrupted text through both. If the free control reproduces QGFD's robustness gap,
the gap was never about diffusion.

**Why E4's controls are fair.** Every iterate is an explicit convex combination
`(1−α)·p⁰ + α·(p⁽ᵗ⁾P)` and `p⁽ᵗ⁾P` is itself a distribution, so
`‖p⁽ᵗ⁾ − p⁰‖_TV ≤ α` holds for *all three* structures — they move the same budget
of probability mass and differ only in destination. `shuffled` additionally permutes
each row's values within the causal prefix, so entropy, sparsity and max-mass are
held fixed and the only variable left is *which key* receives which mass. (Pinned by
`tests/test_mechanism_experiments.py`.)

**What E7 does not test.** A `T` sweep measures the walk's useful *reach*. It does
not establish in-layer *k*-hop composition: on a pretrained checkpoint the key graph
is whatever the checkpoint learned, so that claim is not identifiable here and would
need a from-scratch synthetic-graph model. Note also that `α^T` shrinks
geometrically — at α=0.05 the `T=4` term is 6e-6 of the mass, so a flat curve past
`T=2` is the *expected* outcome, not a bug.

Everything is paired at the token level: one loaded checkpoint per suite, `α`
mutated in place, and byte-identical corrupted strings shared across arms, so there
is no seed variance between the things being compared. Cost: roughly 10–25 min per
model for E2/E3/E4/E6, plus 10–20 min for E7+E8 on the smallest. Set
`RUN_FINETUNE = False` above if you are resuming a session only for this track.

In [ ]:
# --- Track 5: E2, E3, E4, E6, E7, E8 -----------------------------------------
from scripts.mechanism_experiments import (
    DEFAULT_ALPHAS, MECHANISM_EXPERIMENTS, run_mechanism_suite,
)
from scripts.mechanism_experiments import apply_quick as mech_quick

MECH_NOISE_RATE = 0.15     # the headline noise level from Track 1
mechanism = {}

if RUN_MECHANISM:
    for i, mid in enumerate(MODELS):
        which = list(MECH_ALL_MODELS)
        if i == 0:                     # smallest model carries the expensive two
            which += list(MECH_SMALLEST_ONLY)
        short = mid.split("/")[-1]
        cfg = ExperimentConfig(
            model_id=mid, dtype=DTYPE, device="auto",
            diffusion_steps=1, target_alpha=0.05,
            robustness_num_texts=60, attn_num_texts=8,
            latency_seq_len=256, latency_iters=5,
            seed=0, robustness_seed=0, text_sample_seed=0,
            out_dir=f"{RESULTS}/mechanism/{short}",
        )
        if QUICK:
            cfg = mech_quick(cfg)

        print("\n" + "#" * 78)
        print(f"# MECHANISM SUITE — {short}   ({', '.join(which)})")
        print("#" * 78)
        try:
            mechanism[mid] = run_mechanism_suite(
                cfg, which=which, alphas=DEFAULT_ALPHAS,
                noise_rate=MECH_NOISE_RATE, large_model=MECH_LARGE_MODEL,
                depth_steps=(1, 2, 3, 4), quick=QUICK,
                out_dir=f"{RESULTS}/mechanism/{short}",
            )
        except Exception:
            traceback.print_exc()
            print(f"!! mechanism suite FAILED for {mid} — continuing")

print("\nmechanism done:", list(mechanism))

In [ ]:
# --- Track 5 verdict board ---------------------------------------------------
# Reads the JSONs from disk, so it still works after a kernel restart.
import glob, json, os

def _mech_lines(key, r):
    if key == "E2":
        f = (r["clean_curvature"].get("T1") or next(iter(r["clean_curvature"].values())))["fit"]
        g = (r["robustness_by_alpha"] or {}).get("T1", {}).get("by_alpha", {})
        top = max(g, key=float) if g else None
        yield (f"exponent k = {f['exponent_k']:.2f} (predicted 2.0, R2 = {f['r2']:.3f}) "
               f"-> quadratic: {f['consistent_with_quadratic']}"
               if f.get("exponent_k") is not None else f"no fit: {f.get('note')}")
        if top:
            yield f"robustness gap at alpha={float(top):.3f}: {g[top]['robustness_gap_pp']:+.2f} pp"
    elif key == "E3":
        for k, p in r["paired"].items():
            yield (f"alpha={p['alpha']:.3f}: qgfd {p['qgfd_gap_pp']:+.2f} pp vs "
                   f"free temperature (tau={p['matched_tau']:.3f}) {p['temp_gap_pp']:+.2f} pp "
                   f"-> qgfd-temp {p['qgfd_minus_temp_pp']:+.2f} pp | {p['verdict']}")
    elif key == "E4":
        v = r["verdict"] or {}
        yield (f"real {v.get('real_gap_pp', float('nan')):+.2f} pp vs best control "
               f"'{v.get('best_control')}' {v.get('best_control_gap_pp', float('nan')):+.2f} pp "
               f"-> margin {v.get('margin_pp', float('nan')):+.2f} pp")
        yield v.get("reading", "no verdict")
    elif key == "E6":
        for name, row in r["by_corruption"].items():
            tag = "changes tokenisation" if row["changes_tokenisation"] else "tokenisation-preserving"
            yield f"{name:<10} gap {row['robustness_gap_pp']:+.2f} pp   ({tag})"
        yield (r["verdict"] or {}).get("reading", "no verdict")
    elif key == "E7":
        v = r["verdict"] or {}
        yield (f"best T = {v.get('best_T')} at induction {v.get('best_induction_acc')}, "
               f"softmax {v.get('softmax_induction_acc')}, floor {v.get('control_acc_floor')}, "
               f"shape '{v.get('shape')}'")
        yield v.get("reading", "no verdict")
        if v.get("warning"):
            yield "WARNING: " + v["warning"]
    elif key == "E8":
        c = r["comparison"]
        yield (f"QGFD overhead {c['qgfd_overhead_x']:.2f}x | under noise wins: "
               f"{c['winner_under_noise']} | cheaper: {c['latency_cheaper']}")
        yield c["reading"]

for path in sorted(glob.glob(f"{RESULTS}/mechanism/*/mechanism_results.json")):
    with open(path) as fh:
        blob = json.load(fh)
    print("\n" + "=" * 78)
    print(f"{os.path.basename(os.path.dirname(path))}   "
          f"({len(blob['results'])}/{len(blob['requested'])} completed)")
    print("=" * 78)
    for key in blob["requested"]:
        if key in blob["errors"]:
            print(f"  {key}  FAILED  {blob['errors'][key]}")
            continue
        print(f"  {key}  {MECHANISM_EXPERIMENTS[key][0]}")
        for line in _mech_lines(key, blob["results"][key]):
            print(f"        {line}")
        print(f"        a negative result kills: {MECHANISM_EXPERIMENTS[key][1]}")

In [ ]:
# --- E1–E9 coverage preflight ------------------------------------------------
# The question this answers: "did all nine experiments actually produce output?"
# Cell outputs are not evidence — they survive a re-run of an earlier cell and are
# lost on a kernel restart. This walks the FILES on disk using the same two
# discovery functions the report builder uses, so a green board here means the
# report has real data for every row and not a "_Not yet run._" note.
#
# It is a coverage check, not a verdict: E3 can be present and still say QGFD lost.
# Read the verdict board above for what the numbers mean.
import glob, json, os

from scripts.build_report import (_absolute_verdict, _decompose, discover,
                                  discover_mechanism)

_found = discover([RESULTS])
_mech = discover_mechanism([RESULTS])
_zs, _ft, _sy = (_found["zeroshot"], _found["finetune"], _found["synthetic"])

def _mech_present(key):
    """(n_models_carrying_this_experiment, first_failure_message_or_None)."""
    n, err = 0, None
    for path, blob in _mech:
        if key in blob.get("results", {}):
            n += 1
        elif key in blob.get("errors", {}):
            err = err or (f"{blob.get('config', {}).get('model_id', path)}: "
                          f"{blob['errors'][key]}")
    return n, err

_e1 = [json.load(open(p)) for p in glob.glob(f"{RESULTS}/equivalence.json")]
# E5 needs >= 3 seeds AND a paired gap; n=2 is a rehearsal (t_crit = 12.7).
_e5_ok = [a for _, a in _zs
          if a["meta"]["n_seeds"] >= 3 and a.get("paired", {}).get("robustness_gap_pct")]
_e9 = [d for _, a in _zs if (d := _decompose(a))]

rows = [("E1", "alpha=0 bit-exact softmax", len(_e1),
         None if not _e1 or _e1[0]["passed"] else
         f"delta {_e1[0]['max_abs_logit_delta']:.2e} exceeds tolerance"),
        ("E2", "alpha x noise + curvature k", *_mech_present("E2")),
        ("E3", "entropy-matched temperature", *_mech_present("E3")),
        ("E4", "P-structure ablation", *_mech_present("E4")),
        ("E5", "multi-seed paired gap (n>=3)", len(_e5_ok),
         None if _e5_ok or not _zs else
         f"{len(_zs)} zero-shot aggregate(s), none with n>=3 and a paired gap"),
        ("E6", "corruption types", *_mech_present("E6")),
        ("E7", "diffusion depth sweep", *_mech_present("E7")),
        ("E8", "iso-compute vs larger model", *_mech_present("E8")),
        # E9 is derived, not run: it needs Track 1 aggregates that carry per-seed
        # `values` and at least two noise rates. Older aggregates lack `values`.
        ("E9", "denominator control (derived)", len(_e9),
         None if _e9 or not _zs else
         "zero-shot aggregates carry no per-seed `values` — re-run Track 1")]

print(f"{'':<4} {'experiment':<32} {'models':>7}   status")
print("-" * 78)
missing = []
for key, title, n, err in rows:
    if n:
        status = "OK" + (f"   (partial: {err})" if err else "")
    else:
        status = "MISSING" + (f" — {err}" if err else "")
        missing.append(key)
    print(f"{key:<4} {title:<32} {n:>7}   {status}")
print("-" * 78)

for name, n in [("Track 2 fine-tuning A/B (Table 3)", len(_ft)),
                ("Track 3 synthetic probes (Table 4)", len(_sy)),
                ("Track 5 mechanism files (Table 5)", len(_mech)),
                ("overhead vs seq len",
                 int(os.path.exists(f"{RESULTS}/overhead.json"))),
                ("ablation grid",
                 int(os.path.exists(f"{RESULTS}/ablation/ablation.json")))]:
    print(f"     {name:<44} {'OK' if n else 'missing'}  ({n})")

if missing:
    print(f"\n{len(missing)} experiment(s) with no output: {', '.join(missing)}")
    print("Re-run the owning cell before writing prose that depends on them:")
    print("  E1 -> Track 4 equivalence cell | E5 -> Track 1 (needs len(SEEDS) >= 3)")
    print("  E2/E3/E4/E6/E7/E8 -> Track 5 (E7/E8 only run on MODELS[0])")
else:
    print("\nAll of E1-E9 produced output. The report below has data for every row.")

# E9 costs nothing, so print its verdict here rather than only in the report.
if _e9:
    _v = _absolute_verdict(_zs)
    print("\nE9 — is the headline gap a denominator artefact?")
    for d in sorted(_e9, key=lambda d: d["n_params"] or 0):
        print(f"  {d['name']:<22} observed {d['observed']['mean']:+7.2f} pp"
              f" = denominator {d['artefact']['mean']:+7.2f}"
              f" + residual {d['residual']['mean']:+7.2f}"
              f"   abs PPL sm-qgfd {d['abs_gap']['mean']:+8.2f}")
    print({"backwards": "  VERDICT: artefact, and QGFD is significantly WORSE "
                        "in absolute terms. Contribution (2) is falsified.",
           "not_survived": "  VERDICT: the denominator explains most of the gap "
                           "and no residual clears zero.",
           "no_positive": "  VERDICT: no positive residual clears zero — "
                          "nothing established either way.",
           "partly": "  VERDICT: partly artefact — quote the residual.",
           "survives": "  VERDICT: the gap survives the control."}[_v])

## Build the report

`scripts/build_report.py` walks the results tree, ingests every aggregate it finds,
and writes `paper/REPORT.md`. Tracks that were skipped or failed appear as
"_Not yet run._" rather than as blanks, and under-powered runs (n < 3) are called
out in the Threats to Validity section automatically.

In [ ]:
# --- Report ------------------------------------------------------------------
from scripts.build_report import build_report, discover, discover_mechanism

found = discover([RESULTS])
# Track 5 is discovered separately: mechanism_results.json carries a `config` block
# instead of the `meta` block every seed-aggregated file has, and it is single-shot
# rather than aggregated over seeds.
mechanism_files = discover_mechanism([RESULTS])

for track, entries in found.items():
    for path, agg in entries:
        print(f"  [{track}] {agg['meta']['model_id']} n={agg['meta']['n_seeds']}")
for path, blob in mechanism_files:
    print(f"  [mechanism] {blob['config']['model_id']} "
          f"{'+'.join(blob.get('results', {}))}")

text = build_report(found, "paper/REPORT.md", mechanism=mechanism_files)
print(f"\nWrote paper/REPORT.md ({len(text.splitlines())} lines)")

try:
    from IPython.display import Markdown, display
    display(Markdown(text))
except ImportError:
    print(text)

## Before believing any of this — the checklist

1. **Did the patch actually apply?** Every track calls a verifier that raises on a
   no-op patch. A green run is the evidence; a run that printed a
   `GenericAttentionAdapter` warning is not.
2. **Did LoRA reach the live projections?** Track 2 prints
   `LoRA live: N trainable tensors, M with non-zero grad`. At initialisation
   `M ≈ N/2` is correct — `lora_B` starts at zero, so `lora_A` gets no gradient
   on the first probe. `M = 0` would have raised.
3. **Did α warm up?** The `alpha` block in each fine-tuning result must show
   `step_count ≥ warmup_steps` and `alpha_train_mode == target_alpha`.
4. **Is n ≥ 3?** With n=2 the t-critical value is 12.7 and essentially nothing can
   reach significance. Two seeds is a rehearsal, not a result.
5. **Read the paired column, not the per-arm columns.** Between-seed corpus
   variance is far larger than the effect; only the within-seed difference has the
   resolution to say anything.
6. **α=0 equivalence must pass (E1).** If it does not, the drop-in claim is false
   and the rest of the report is meaningless.
7. **Did E3 survive?** A robustness gap that an entropy-matched temperature rescale
   also produces is not a QGFD result — temperature is free and folds into `W_Q`.
   Check `qgfd_minus_temp_pp > 0` on the verdict board before claiming a mechanism.
8. **Did E4 survive?** If `uniform` or `shuffled` P matches `real` P at the same α,
   report the effect as generic probability-mass smoothing and drop "graph" from the
   framing. All three move the same mass, so this is not a power question.
9. **Is the gap in the right units?** A `+12 pp` gap against a baseline that
   degraded by `1500%` is under 1% of the damage. Report both the gap and the
   degradation it is a fraction of, or the number reads as far larger than it is.
10. **Is the gap a denominator artefact (E9)?** `robustness_gap_pct` divides by
    each arm's own clean perplexity, so QGFD's clean-PPL cost enlarges its
    denominator and shrinks its Δ% for free. Table 1b splits the gap into that
    arithmetic and a residual. Only the residual — equivalently, the sign of the
    absolute perplexity difference under noise — can be called robustness. This
    check costs nothing and it outranks the raw gap.
11. **Does the effect survive scale?** Compare the paired gap across the three
    models. A gap that shrinks with parameter count, or reverses on the largest
    model, is a small-model artefact and must be reported as one.